## gCRL-AE applied on a simple simulated dataset

- Load example adata (with TF communities already specified)
- Compute eigengenes
- Train gCRL-AE
- Partial MCC permutation analysis

In [1]:
# ensuring all packages are reloaded each time I run a cell
%load_ext autoreload
%autoreload 2

In [2]:
# importing
import scanpy as sc
from gcrl.training.train_gcrl_ae import train_gcrl_ae
from gcrl.grn.eigengenes import compute_eigengenes
from gcrl.alignment.partial_mcc_perm_experiments import run_partial_mcc_perm_experiments

In [3]:
# loading data
adata = sc.read_h5ad("../../data/example/adata.h5ad")
adata

AnnData object with n_obs × n_vars = 2000 × 1087
    obs: 'intervention', 'cell_type', 'intervention_comm', 'set'
    var: 'kind', 'community'

In [4]:
adata.obs

,intervention,cell_type,intervention_comm,set
unperturbed_0,unperturbed,1,NaN,training
unperturbed_1,unperturbed,1,NaN,training
unperturbed_2,unperturbed,1,NaN,training
unperturbed_3,unperturbed,1,NaN,training
unperturbed_4,unperturbed,1,NaN,training
...,...,...,...,...
pert_c4_195,TF_c4_n15,1,4.0,training
pert_c4_196,TF_c4_n15,1,4.0,training
pert_c4_197,TF_c4_n15,1,4.0,training
pert_c4_198,TF_c4_n15,1,4.0,training


In [5]:
# computing eigengenes
compute_eigengenes(adata)
adata

AnnData object with n_obs × n_vars = 2000 × 1087
    obs: 'intervention', 'cell_type', 'intervention_comm', 'set'
    var: 'kind', 'community'
    uns: 'X_comm_eig_comm_ids', 'X_comm_eig_global_index', 'comm_eig_meta'
    obsm: 'X_comm_eig'

In [6]:
# training gCRL-AE
res = train_gcrl_ae(
    adata=adata,
    input_mode="TF",                  # encoder sees TFs only (default)
    reconstruct_all=True,             # decoder reconstructs all genes
    latent_dim=None,                  # inferred as (#communities + 1) if None
    standardize="zscore_ref",         # z-score using unperturbed cells as reference (default)
    reference_query='intervention == "unperturbed"',
    num_epochs=100,
    lr=1e-3,
    device=None,                      # auto-select cuda if available
    outdir="../../results/simulated/example/ae",   # optional: save artifacts
)

In [7]:
# embeddings
B = res.embeddings              # (n_cells, latent_dim)
print(B.shape)
B

(2000, 6)


array([[-2.0946362 , -1.590497  ,  0.63113666, -0.3340325 , -0.33104116,
         0.11999617],
       [-0.71722424,  0.14230996,  0.99107414,  0.7043285 , -0.8965558 ,
        -1.3013563 ],
       [ 0.6829108 ,  0.02633343, -1.0635486 ,  0.11595932,  1.2184563 ,
        -1.1155654 ],
       ...,
       [ 2.79285   ,  0.22033162, -0.31682163, -2.4279966 ,  0.22451667,
         2.5713148 ],
       [ 0.43668678, -1.2959898 , -1.6084077 , -1.9586792 , -2.079186  ,
        -0.62547857],
       [ 2.0507967 , -2.017263  ,  2.060498  ,  0.37142363, -1.9463701 ,
         0.40096486]], dtype=float32)

In [ ]:
# partial mcc permutation
out = run_partial_mcc_perm_experiments(
        adata=adata,
        embeddings=B,
        community_col='community',
        reference_query='intervention == "unperturbed"',
        mode="by_cell_type",
        cell_type_col="cell_type",
        n_real_seeds=5,
        n_permutations=3,
        n_perm_seeds=2,
        lr=5e-2,
        steps=100,
        device="cpu",
        master_seed=123,
        save_density_path='../../results/simulated/example/partial_mcc_permutation/dens.png',
    )
print(out["scores_real"])
print(out["scores_perm"])